In [ ]:
import sys
sys.executable

In [ ]:
# STEP 1: IMPORT QUALTRICS DATA VIA API
import requests
import zipfile
import io
import paramiko
import os
import dtale
import pandas as pd
from pandasgui import show
import time # Import time for polling delay

# -------------------------------------
API_TOKEN = "xapt4GarlMnBKLM0aR0nypi3wWpHlAoTmfuvIro3"
DATA_CENTER = "ca1"
SURVEY_ID = "SV_9BMGubXPrEQywYu"
FILE_FORMAT = "csv"

# -------------------------------------

BASE_URL = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/export-responses"

headers = {
    "content-type": "application/json",
    "x-api-token": API_TOKEN
}

# Request export with labels (Simplified to prevent 400 error)
# **Crucial Change**: We keep useLabels: True to get the string labels.
payload = {
    "format": FILE_FORMAT,
    "useLabels": True
    # Removed "includeVariableNames": True for stability
}

# Create Export
create_resp = requests.post(
    BASE_URL,
    json=payload, # **Crucial Change**: Using the full payload here
    headers=headers
)
create_resp.raise_for_status() # Check for errors like 400 or 500
progress_id = create_resp.json()["result"]["progressId"]

print(f"Export started. Progress ID: {progress_id}")

# Poll Until File is Ready
while True:
    check_resp = requests.get(f"{BASE_URL}/{progress_id}", headers=headers)
    check_resp.raise_for_status()

    status = check_resp.json()["result"]["status"]
    print("Status:", status)

    if status == "complete":
        file_id = check_resp.json()["result"]["fileId"]
        break
    elif status == "failed":
        raise Exception("Qualtrics export failed.")
    
    # Wait for a few seconds before checking status again
    time.sleep(5) 

# Download using fileId
download_resp = requests.get(
    f"{BASE_URL}/{file_id}/file",
    headers=headers,
    stream=True
)
download_resp.raise_for_status()

# Extract ZIP file and load CSV into Pandas
zipped = zipfile.ZipFile(io.BytesIO(download_resp.content))
csv_filename = zipped.namelist()[0]
df = pd.read_csv(zipped.open(csv_filename))

print("Downloaded", csv_filename, "with", len(df), "rows.")

# Save as Excel file
output_path = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 visa survey MOST RECENT QUALTRICS EXTRACT.xlsx"
df.to_excel(output_path, index=False)

print("Saved Excel file to:", output_path)

In [ ]:
# STEP 2: IMPORT EXTRACTED QUALTRICS DATA, CLEAN, AND SAVE CLEANED DATA FILE IN SHAREPOINT
import smtplib
from email.message import EmailMessage
import bokeh
import os
from pandasgui import show

# Clean the Excel file
VisaSurveyRawData = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 visa survey MOST RECENT QUALTRICS EXTRACT.xlsx"
VisaSurveyDataCleaned = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 Visa Survey CLEANED Qualtrics data.xlsx"

# Import Excel
VisaSurveyDataDF = pd.read_excel(VisaSurveyRawData, header=0)

# Clean the dataframe (use VisaSurveyDataDF, not VisaSurveyRawData!)
VisaSurveyDataDF = VisaSurveyDataDF.iloc[2:].reset_index(drop=True)

VisaSurveyDataDF.rename(columns={'Status': 'Source'}, inplace=True)
VisaSurveyDataDF.rename(columns={'Q1': 'Status from Qualtrics'}, inplace=True)
VisaSurveyDataDF.rename(columns={'Will apply': 'Consular Post Will Apply To'}, inplace=True)
VisaSurveyDataDF.rename(columns={'Did apply': 'Consular Post Already Applied To'}, inplace=True)
VisaSurveyDataDF.rename(columns={'Bio Data_4': 'StudentID'}, inplace=True)
VisaSurveyDataDF['StudentID'] = VisaSurveyDataDF['StudentID'].str.replace(
    r'^u', 'U', regex=True
)
VisaSurveyDataDF['StudentID'] = (VisaSurveyDataDF['StudentID'].replace('U62597668', 'U62597669'))
VisaSurveyDataDF = VisaSurveyDataDF.sort_values(by='RecordedDate').reset_index(drop=True)
VisaSurveyDataDF = VisaSurveyDataDF.drop_duplicates(subset='StudentID', keep='first')

# Save cleaned file
VisaSurveyDataDF.to_excel(VisaSurveyDataCleaned, index=False)
print("Data cleaned and saved successfully!")
dtale.show(VisaSurveyDataDF)


In [ ]:
# STEP 3: EXTRACT ISSO VISA DATA FROM SSH FILE
import paramiko
import io
import os
import dtale

# Connection details
host = "your.university.host.edu"
username = "your_student_id"
# Point this to the private key file you just generated
key_path = os.path.expanduser("~/.ssh/id_rsa") 

client = paramiko.SSHClient()
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Load the private key
    private_key = paramiko.RSAKey.from_private_key_file(key_path)
    
    # Connect using the key instead of password
    client.connect(host, username=username, pkey=private_key)
    
    sftp = client.open_sftp()
    
    remote_path = "/path/to/your/data/visa_stats.csv"
    with sftp.open(remote_path, 'r') as remote_file:
        RawVisaData = pd.read_csv(remote_file)
    
    print("Success! Data ingested via SSH Key.")
    print(RawVisaData.head())

except Exception as e:
    print(f"Error: {e}")

finally:
    if 'sftp' in locals(): sftp.close()
    client.close()

In [ ]:
# STEP 4: IMPORT MANUAL UPDATE FILE FROM SHAREPOINT
#Import data
ManUpdatesRawData= r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 Manual Visa Updates.xlsx"
ManUpdatesDF= pd.read_excel(ManUpdatesRawData, header=0)
#Rename columns
ManUpdatesDF.rename(columns={'Revised Visa Status': 'Final Visa Status from Manual Updates'}, inplace=True)
dtale.show(ManUpdatesDF)

In [ ]:
NewGrads = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 Entering International Graduate Students.xlsx"
DFNewGrads=pd.read_excel(NewGrads, header=0)
DFNewGrads.rename(columns={"Unnamed: 1": "TermType"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 2": "TermNumber"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 5": "LastName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 6": "MiddleName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 7": "FirstName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 24": "CampusLocationShort"}, inplace=True)
dtale.show(DFNewGrads)

In [ ]:
# === STEP 5: Import and clean FA26 entering master's students and ISSO data
import numpy as np
#Set path name for each file
NewGrads = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 Entering International Graduate Students.xlsx"
VisaStatuses = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Most Recent ISSO Data for Fall 2026.xlsx"
MergedFiles = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 new masters students and visa statuses MERGED.xlsx"
ManualUpdates = r"C:\Users\iabaker\Boston University\Provost Graduate Enrollment - Documents\Admissions & Operations\Fall 2026 International Students Visa Survey\Fall 2026 Manual Visa Updates.xlsx"
#Import and initial cleaning of NewGrads data file
DFNewGrads=pd.read_excel(NewGrads, header=0)
DFNewGrads.rename(columns={"Unnamed: 1": "TermType"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 2": "TermNumber"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 5": "LastName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 6": "MiddleName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 7": "FirstName"}, inplace=True)
DFNewGrads.rename(columns={"Unnamed: 24": "CampusLocationShort"}, inplace=True)
DFNewGrads.rename(columns={'Person Id': 'StudentID'}, inplace=True)
duplicates = DFNewGrads[DFNewGrads.duplicated(subset='StudentID', keep=False)]
DFNewGrads = DFNewGrads.drop_duplicates(subset='StudentID', keep='first')
DFNewGrads.loc[
    DFNewGrads['Graduate Application Academic Program'].str.contains('Pardee', na=False),
    'Graduate Application College'
] = 'Pardee School of Global Studies'
#Import and initial cleaning of VisaStatuses file from ISSO
DFVisaStatuses=pd.read_excel(VisaStatuses, header=0)
DFVisaStatuses.rename(
    columns={
        'BUID': 'StudentID',
        'First Name': 'FirstName',
        'Last Name': 'LastName'
    },
    inplace=True)
DFVisaStatuses = DFVisaStatuses.sort_values(by='StudentID').reset_index(drop=True)
duplicates2 = DFVisaStatuses[DFVisaStatuses.duplicated(subset='StudentID', keep=False)]
#Notify that all steps were completed successfully
print("All steps executed successfully!")
dtale.show(DFVisaStatuses)

In [ ]:
# === STEP 6: Merge NewGrads and VisaStatuses, then complete variable cleaning on the combined dataframe
#Merge NewGrads and VisaStatuses data, create a Final Visa Status column and a Visa Status Source Column, and reorder columns
ManUpdatesDF=pd.read_excel(ManualUpdates, header=0)
merged_df=pd.merge(DFNewGrads, DFVisaStatuses, on='StudentID', how='outer', indicator=True)
merged_df.rename(columns={'_merge': 'ADWISSOmerge'}, inplace=True)
merged_df['Final Visa Status'] = np.nan
merged_df['Final Visa Status Source'] = np.nan
front_cols = ['ADWISSOmerge', 'Applicant Visa', 'I-901 Transaction Date(Student)', 'Visa Issue Date', 'Final Visa Status', 'Final Visa Status Source']
other_cols = [c for c in merged_df.columns if c not in front_cols]
merged_df = merged_df[front_cols + other_cols]
#Drop observations for students who deferred
merged_df = merged_df.sort_values(by=['ADWISSOmerge', 'StudentID'])
merged_df = merged_df[~merged_df['StudentID'].isin(['U54759000', 'U97848185'])]
merge_counts = merged_df['ADWISSOmerge'].value_counts()
#Update FinalVisaStatus and FinalVisaStatusSource column for students w/ a confirmed visa from ISSO
merged_df['Final Visa Status'] = np.where(
    merged_df['Visa Issue Date'].notna(),
    "I have already received my student visa for my studies at BU.",
    merged_df['Final Visa Status']
)
merged_df['Final Visa Status Source'] = np.where(
    merged_df['Visa Issue Date'].notna(),
    "ISSO data",
    merged_df['Final Visa Status Source']
)
#Notify that all steps were completed successfully
print("All steps executed successfully!")
dtale.show(merged_df)

In [ ]:
# === STEP 7: Merge in Qualtrics data and manual updates data
#Merge in Qualtrics data and reorder columns
merged_df_2=pd.merge(merged_df, VisaSurveyDataDF, on='StudentID', how='outer', indicator=True)
merged_df_2.rename(columns={'_merge': 'QualtricsMerge'}, inplace=True)
front_cols_2 = ['Final Visa Status', 'Final Visa Status Source', 'ADWISSOmerge', 'QualtricsMerge', 'Applicant Visa', 'I-901 Transaction Date(Student)', 'Visa Issue Date', 'Status from Qualtrics']
other_cols_2 = [c for c in merged_df_2.columns if c not in front_cols_2]
merged_df_2 = merged_df_2[front_cols_2 + other_cols_2]
#Update Final Visa Status with Qualtrics data for students who don't have it from ISSO
cond = (
    (merged_df_2['Final Visa Status'].isna()) | (merged_df_2['Final Visa Status'] == 'nan')) & (merged_df_2['QualtricsMerge'] == 'both') & (merged_df_2['Status from Qualtrics'].notna())
merged_df_2.loc[cond, 'Final Visa Status'] = merged_df_2.loc[cond, 'Status from Qualtrics']
merged_df_2.loc[cond, 'Final Visa Status Source'] = 'Qualtrics'
#Merge with manual updates
merged_df_3=pd.merge(merged_df_2, ManUpdatesDF, on='StudentID', how='left', indicator=True)
merged_df_3.rename(columns={'_merge': 'ManUpdatesMerge'}, inplace=True)
#Notify that all steps were completed successfully
print("All steps executed successfully!")
dtale.show(merged_df_3)

In [ ]:
# === STEP 8: Complete data file cleaning
#Update FinalVisaStatus with manual updates if the student doesn't have a confirmed visa from ISSO and does have an update in the manaul updates file
cond = (
    (merged_df_3['ManUpdatesMerge'] == 'both')
)
merged_df_3.loc[cond, 'Final Visa Status'] = merged_df_3.loc[cond, 'Revised Visa Status']
merged_df_3.loc[cond, 'Final Visa Status Source'] = "Manual updates"
#Update FinalVisaStatus to indicate no updates are available if it's still null
merged_df_3['Final Visa Status'] = merged_df_3['Final Visa Status'].replace([np.nan, None, 'nan'], "No updates from ISSO, Qualtrics, or colleges")
#Clean FirstName and LastName fields
merged_df_3.rename(columns={
    'LastName_x': 'LastName',
    'FirstName_x': 'FirstName',
    'LastName_y': 'LastName_ISSO',
    'FirstName_y': 'FirstName_ISSO'
}, inplace=True)
merged_df_3['LastName'] = merged_df_3['LastName'].fillna(merged_df_3['LastName_ISSO'])
merged_df_3['FirstName'] = merged_df_3['FirstName'].fillna(merged_df_3['FirstName_ISSO'])
#Selet columns to keep
keep_cols = ['Final Visa Status', 'Final Visa Status Source', 'ADWISSOmerge', 
                'QualtricsMerge', 'ManUpdatesMerge', 'Applicant Visa', 'I-901 Transaction Date(Student)', 
                'Visa Issue Date', 'Status from Qualtrics', 'Consular Post Will Apply To', 'Consular Post Already Applied To',
             'StudentID', 'LastName', 'FirstName','Applicant Country of Citizenship',
             'Graduate Application Degree Level','Graduate Application College', 'Graduate Application Academic Program', 'Graduate Application Major','Graduate Application Program Campus Location',
            'Email', 'Graduate Decision Status', 'Visa Type', 'Revised Visa Status', 'Manual Update', 'Manual Update Date']
merged_df_3 = merged_df_3[keep_cols]
merged_df_3 = merged_df_3[ merged_df_3['Final Visa Status'] != 'N/A-Exclude from Dashboard' ]
#Reorder columns
front_cols_3 = ['StudentID', 'FirstName', 'LastName', 'Final Visa Status', 'Final Visa Status Source', 'ADWISSOmerge', 
                'QualtricsMerge', 'ManUpdatesMerge', 'Applicant Visa', 'I-901 Transaction Date(Student)', 
                'Visa Issue Date', 'Status from Qualtrics', 'Consular Post Will Apply To', 'Consular Post Already Applied To']
other_cols_3 = [c for c in merged_df_3.columns if c not in front_cols_3]
merged_df_3 = merged_df_3[front_cols_3 + other_cols_3]
#Export to Excel
merged_df_3.to_excel(MergedFiles, index=False)
#Notify that all steps were completed successfully
print("All steps executed successfully!")
dtale.show(merged_df_3)

In [ ]:
# STEP 9: SEND AN EMAIL INDICATING THAT ALL STEPS IN THE PYTHON CODE HAVE BEEN RUN CORRECTLY ===
import win32com.client as win32

# Connect to Outlook
outlook = win32.Dispatch('outlook.application')

mail = outlook.CreateItem(0)
mail.To = "iabaker@bu.edu; kjmarco@bu.edu"
mail.Subject = "Visa Survey Data Update Completed"
mail.Body = (
    "Hello,\n\n"
    "This is an automated notification that the Fall 2026 Visa Survey data "
    "has been cleaned and the cleaned Excel file has been updated successfully.\n\n"
    "Cheers,\n"
    "Ian"
)
mail.Send()

print("Email sent successfully via Outlook client!")